# Benchmark 3/3 — Mistake Detection

Component tests for the **MistakeDetector** and the end-to-end app mistake pipeline
(flags substitutions / insertions / deletions vs the score).

No public dataset ships note-level *mistake* labels for our setup, so we
**synthesise errors with known ground truth** via `MistakeInjector`, following
*Polytune* (Chou et al., "Detecting Music Performance Errors with Transformers",
AAAI 2025): each note gets an error with prob λ, mapped to wrong-pitch /
missed / extra ⇒ our substitution / deletion / insertion.

Two evals:
1. **Symbolic** — perturbed notes fed straight to the MistakeDetector (isolates the
   aligner), `mode="symbolic"`. Run isolated-per-type *and* mixed. Pass
   `correct_symbolic=True` only when you explicitly want to include MistakeChecker
   correction on symbolic notes.
2. **End-to-end** — render the perturbed performance, run the full pipeline,
   `mode="audio"`; plus `clean_render_mistakes` (faithful render ⇒ ≈0 mistakes).

> Public alternatives considered: *MAESTRO-E* / *CocoChorales-E*
> (github.com/ben2002chou/Polytune) are released but piano/multi-instrument and
> 200–300+ hrs — off-instrument and too large for a component test, so we generate
> violin-specific errors on the etude corpus, reusing their recipe. Rendered
> error-injected etudes are written to `mistake-injected-violin-etudes/`.

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import sys
sys.path.insert(0, "../benchmarks")
from MistakeBenchmarker import MistakeBenchmarker, MistakeInjector
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
bm = MistakeBenchmarker(max_tracks=5)

## Eval 1 — symbolic injection (pure MistakeDetector)
Isolated-per-type is the detector's ceiling on each error alone; **mixed** is the
realistic case where types interact (and edit-cost choices can cascade).
Macro-averaged over both etude sets.

In [ ]:
def scenario_overall(weights, mode="symbolic", **kw):
    res = [bm.bench_mistake_dataset(ds, MistakeInjector(mistake_rate=0.2, weights=weights),
                                    seeds=range(6), mode=mode, verbose=False, **kw)
           for ds in bm.ETUDE_DATASETS]
    return pd.concat(res).groupby(level=0).mean().loc["OVERALL"]

scenarios = {"sub-only": (1,0,0), "del-only": (0,1,0),
             "ins-only": (0,0,1), "mixed": (0.4,0.3,0.3)}
pd.DataFrame({name: scenario_overall(w) for name, w in scenarios.items()}).T

Per-type detail for the realistic **mixed** scenario:

In [ ]:
mixed = bm.bench_mistake_dataset("kayser", MistakeInjector(mistake_rate=0.2, weights=(0.4,0.3,0.3)),
                                 seeds=range(6), mode="symbolic", verbose=False, write=True)
mixed

> **Reading it:** isolated types are usually easier than mixed cases. Mixed precision
> is where `ins_cost` / `del_cost`, the distance-weighted substitution cost, and
> `pitch_tolerance` interact; this benchmark is the knob to tune those tradeoffs.

## Eval 2 — end-to-end (audio) + clean-render false-positive
Render the perturbed performance and run the **full pipeline**; heavier, so a few
files / short excerpt. The clean render (no injection) should give ≈0 mistakes.

In [ ]:
e2e_scores, clean = [], []
injector = MistakeInjector(mistake_rate=0.2)
for ds in bm.ETUDE_DATASETS:
    title, mid = next(bm.iter_etudes(ds))
    e2e_scores.append(bm.bench_mistake_track(mid, injector, seeds=range(2), mode="audio", max_sec=20))
    clean.append(bm.clean_render_mistakes(mid, max_sec=20))

print("end-to-end injected (onset-matched, per file):")
for ds, df in zip(bm.ETUDE_DATASETS, e2e_scores):
    print(f"  {ds}: OVERALL F-measure={df.loc['OVERALL','F-measure']:.3f}")
display(pd.concat(e2e_scores, keys=bm.ETUDE_DATASETS))
print("\nclean-render false positives (lower is better):")
pd.DataFrame(clean)

## References
* **Methodology:** Chou, Yang, Tsai, Dannenberg, Su — *Detecting Music Performance
  Errors with Transformers*, AAAI 2025 (arxiv.org/abs/2501.02030).
* **Public error datasets** (not used; see header): MAESTRO-E, CocoChorales-E —
  github.com/ben2002chou/Polytune.
* The **symbolic** eval is the true MistakeDetector component test (perfect note
  input); **end-to-end** folds in pitch/note-detection error, so it's a lower
  bound. Very short inserted notes can be merged away by note detection — a real
  pipeline limitation surfaced here.